## Interval Analysis

In [ ]:
import torch
print(torch.__version__)
print(torch.__file__)

In [ ]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = False
device = torch.device("mps")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200,10)

    def forward(self, x):
        x = x.view((-1, 28*28))
        x = F.relu(self.fc(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=-1) # added softmax for probabilities
        return x

class Normalize(nn.Module):
    def forward(self, x):
        return (x - 0.1307)/0.3081

# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = nn.Sequential(Normalize(), Net())

model = model.to(device)
model.train()


In [ ]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')

In [19]:
train_model(model, 15)
test_model(model)

Epoch 1/15, Loss: 1.538
Epoch 2/15, Loss: 1.536
Epoch 3/15, Loss: 1.535
Epoch 4/15, Loss: 1.533
Epoch 5/15, Loss: 1.531
Epoch 6/15, Loss: 1.530
Epoch 7/15, Loss: 1.529
Epoch 8/15, Loss: 1.527
Epoch 9/15, Loss: 1.526
Epoch 10/15, Loss: 1.525
Epoch 11/15, Loss: 1.524
Epoch 12/15, Loss: 1.523
Epoch 13/15, Loss: 1.521
Epoch 14/15, Loss: 1.520
Epoch 15/15, Loss: 1.520
Accuracy on images: 94.71


### Write the interval analysis for the simple model

In [ ]:
# !pip install bound-propagation

In [ ]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

import torch
import torch.nn as nn
from bound_propagation import BoundModelFactory, HyperRectangle


model.eval()
# For non customized layer
interval_model = nn.Sequential(
    model[1].fc,
    nn.ReLU(),
    model[1].fc2,
)

factory = BoundModelFactory()
bound_model = factory.build(interval_model)

def push_interval(images, labels, eps):
    # For customized layer .... who decided this ....
    lower = (torch.clamp(images - eps, 0, 1) - 0.1307) / 0.3081
    upper = (torch.clamp(images + eps, 0, 1) - 0.1307) / 0.3081

    lower = lower.flatten(1)
    upper = upper.flatten(1)

    bounds1 = HyperRectangle(lower, upper)
    bounds2 = bound_model.ibp(bounds1)

    # "Output layer" where we determine fy (x′)−fj (x′) for each j ̸= y.
    # fy(x)
    fy_x = bounds2.lower.gather(1, labels.unsqueeze(1))

    # computing fy(x) - fj(x)
    differences = fy_x - bounds2.upper

    # we need to account for the fact that the previous line does this for y = j. replace with +inf.
    differences = differences.scatter(1, labels.unsqueeze(1), float('inf'))

    # determine if all non-negatuve
    verified = (differences >= 0).all(dim=1)

    return verified


def test_interval(eps):

    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            verified = push_interval(images, labels, eps)
            total += labels.size(0)
            correct += verified.sum().item()
        print(f'Epsilon: {eps}. Verified accuracy on images: {100 * correct / total}')

for eps in torch.linspace(0.01, 0.1, 10):
    test_interval(eps)



Epsilon: 0.009999999776482582. Verified accuracy on images: 50.58
Epsilon: 0.019999999552965164. Verified accuracy on images: 5.24
Epsilon: 0.030000001192092896. Verified accuracy on images: 0.25
Epsilon: 0.04000000283122063. Verified accuracy on images: 0.0
Epsilon: 0.05000000447034836. Verified accuracy on images: 0.0
Epsilon: 0.05999999865889549. Verified accuracy on images: 0.0
Epsilon: 0.07000000029802322. Verified accuracy on images: 0.0
Epsilon: 0.07999999821186066. Verified accuracy on images: 0.0
Epsilon: 0.09000000357627869. Verified accuracy on images: 0.0
Epsilon: 0.10000000149011612. Verified accuracy on images: 0.0


In [21]:
test_interval(0.0)

Epsilon: 0.0. Verified accuracy on images: 94.71
